# 🏢 Employee Attrition Prediction
**Internship Project — Week 2 | XylofyAI**  
**Author:** Shreyas Yarabadi  
**Dataset:** IBM HR Analytics Employee Attrition Dataset (1,470 records, 35 features)  

---

**Goal:** Build a machine learning system that predicts whether an employee is likely to leave the company, and extract actionable HR insights from the results.

**Tasks covered:**
1. Data Loading & Exploration
2. Data Cleaning & Preprocessing
3. Exploratory Data Analysis (EDA)
4. Model Building & Comparison
5. Model Evaluation
6. Visualizations
7. HR Insights & Business Recommendations

---
## Task 1 — Data Loading & Exploration

In [ ]:
# ── Package Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print("All packages imported successfully.")

In [ ]:
# ── Load Dataset via KaggleHub ────────────────────────────────────────────────
# The dataset is loaded directly from Kaggle using the kagglehub adapter.
# This eliminates the need to manually download the CSV.

import kagglehub
from kagglehub import KaggleDatasetAdapter

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "pavansubhasht/ibm-hr-analytics-attrition-dataset",
    "WA_Fn-UseC_-HR-Employee-Attrition.csv",
)

print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns")

In [ ]:
# ── First 10 Rows ────────────────────────────────────────────────────────────
print("First 10 rows of the dataset:")
df.head(10)

In [ ]:
# ── Shape & Column Types ─────────────────────────────────────────────────────
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")

numeric_cols = df.select_dtypes(include='number').columns.tolist()
categorical_cols = df.select_dtypes(exclude='number').columns.tolist()

print(f"Numeric columns  : {len(numeric_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")
print(f"\nCategorical columns: {categorical_cols}")

In [ ]:
# ── Target Column: Attrition ─────────────────────────────────────────────────
# Count how many employees left (Yes) vs stayed (No)

attrition_counts = df['Attrition'].value_counts()
attrition_rate   = attrition_counts['Yes'] / len(df) * 100

print(f"Employees who LEFT  (Yes): {attrition_counts['Yes']}")
print(f"Employees who STAYED (No): {attrition_counts['No']}")
print(f"\nAttrition Rate: {attrition_rate:.2f}%")
print()
print("""
Observation:
  Only ~16% of employees left, while 84% stayed.
  This is a classic imbalanced classification problem —
  a naive model that always predicts 'No' would be 84% accurate,
  yet completely useless for HR purposes. We must account for
  this imbalance during model training.
""")

---
## Task 2 — Data Cleaning & Preprocessing

In [ ]:
# ── Check for Missing / Null Values ──────────────────────────────────────────
null_counts = df.isnull().sum()

if null_counts.sum() == 0:
    print("No missing values found. Dataset is clean.")
else:
    print("Columns with missing values:")
    print(null_counts[null_counts > 0])

In [ ]:
# ── Drop Irrelevant / Constant Columns ───────────────────────────────────────
# These columns carry no predictive signal:
#   EmployeeNumber   — just a unique ID
#   EmployeeCount    — always 1
#   Over18           — always 'Y'
#   StandardHours    — always 80

cols_to_drop = ['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']
df = df.drop(columns=cols_to_drop)

print(f"Dropped columns: {cols_to_drop}")
print(f"Remaining columns: {df.shape[1]}")

In [ ]:
# ── Encode Target Column: Attrition → 1 / 0 ──────────────────────────────────
# Yes → 1 (left the company)
# No  → 0 (stayed)

df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

print("Attrition encoding complete:")
print(df['Attrition'].value_counts())

In [ ]:
# ── One-Hot Encode Remaining Categorical Columns ──────────────────────────────
# Converts text categories like 'Sales', 'Male', 'Single' into numeric dummy columns.
# drop_first=True avoids multicollinearity (dummy variable trap).

categorical_features = df.select_dtypes(exclude='number').columns.tolist()
print(f"Encoding {len(categorical_features)} categorical columns: {categorical_features}")

df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

print(f"\nDataframe shape after encoding: {df_encoded.shape}")

In [ ]:
# ── Feature / Target Split & Standard Scaling ─────────────────────────────────
# StandardScaler transforms each numeric feature to mean=0, std=1.
# This is important for Logistic Regression which is sensitive to feature scale.

X = df_encoded.drop(columns=['Attrition'])
y = df_encoded['Attrition']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(f"Features (X): {X_scaled.shape[1]} columns")
print(f"Target  (y): {y.shape[0]} rows")
print(f"\nPreprocessing complete — data is ready for modelling.")

---
## Task 3 — Exploratory Data Analysis (EDA)

We use the **original unencoded dataframe** (`df` before one-hot encoding) for EDA, since readable labels are easier to interpret in charts.

In [ ]:
# ── EDA Setup: Reload original df with readable labels ────────────────────────
# We use 'df' (post-drop, pre-encoding) which still has text categories.
# Attrition is already encoded as 1/0 — mean() gives us the attrition rate directly.

print("Using df for EDA (post-drop, binary Attrition, text categories)")
print(f"Shape: {df.shape}")

In [ ]:
# ── Attrition Rate by Department ─────────────────────────────────────────────
dept_attrition = df.groupby('Department')['Attrition'].mean().sort_values(ascending=False) * 100

print("Attrition Rate by Department:")
for dept, rate in dept_attrition.items():
    print(f"  {dept:<30} {rate:.1f}%")

In [ ]:
# ── Attrition Rate by Job Role ────────────────────────────────────────────────
role_attrition = df.groupby('JobRole')['Attrition'].mean().sort_values(ascending=False) * 100

print("Attrition Rate by Job Role:")
for role, rate in role_attrition.items():
    print(f"  {role:<35} {rate:.1f}%")

In [ ]:
# ── Attrition vs Monthly Income ───────────────────────────────────────────────
income_left   = df[df['Attrition'] == 1]['MonthlyIncome'].mean()
income_stayed = df[df['Attrition'] == 0]['MonthlyIncome'].mean()

print(f"Avg Monthly Income — Employees who LEFT  : ${income_left:,.0f}")
print(f"Avg Monthly Income — Employees who STAYED: ${income_stayed:,.0f}")
print(f"\nGap: ${income_stayed - income_left:,.0f} more per month for those who stayed.")

In [ ]:
# ── Attrition vs Work-Life Balance ────────────────────────────────────────────
# Rating scale: 1=Bad, 2=Good, 3=Better, 4=Best

wlb_attrition = df.groupby('WorkLifeBalance')['Attrition'].mean() * 100

labels = {1: 'Bad', 2: 'Good', 3: 'Better', 4: 'Best'}
print("Attrition Rate by Work-Life Balance Rating:")
for rating, rate in wlb_attrition.items():
    print(f"  Rating {rating} ({labels[rating]:<6}): {rate:.1f}%")

In [ ]:
# ── Attrition vs Years at Company ────────────────────────────────────────────
# Group employees by tenure band and compute attrition rate for each band.

bins   = [0, 2, 5, 10, 20, 40]
labels_tenure = ['0–2 yrs', '3–5 yrs', '6–10 yrs', '11–20 yrs', '20+ yrs']

df['TenureBand'] = pd.cut(df['YearsAtCompany'], bins=bins, labels=labels_tenure)
tenure_attrition = df.groupby('TenureBand', observed=True)['Attrition'].mean() * 100

print("Attrition Rate by Tenure Band:")
for band, rate in tenure_attrition.items():
    print(f"  {band:<12}: {rate:.1f}%")

# Clean up — drop helper column
df.drop(columns=['TenureBand'], inplace=True)

In [ ]:
# ── 5 Key EDA Business Insights ──────────────────────────────────────────────
print("""
========== EDA Business Insights ==========

1. SALES DEPARTMENT LEADS IN ATTRITION:
   The Sales department has the highest attrition rate (~21%),
   nearly double that of Research & Development (~14%).
   HR should prioritise retention efforts here.

2. SALES REPRESENTATIVES ARE THE HIGHEST RISK ROLE:
   Sales Representatives have ~40% attrition — by far the
   highest of all job roles. This is a critical pipeline risk
   given the cost of replacing sales staff.

3. LOWER INCOME = HIGHER EXIT RISK:
   Employees who left earned on average ~$2,000 less per month
   than those who stayed. However, income alone does not explain
   all attrition — other factors are also at play.

4. POOR WORK-LIFE BALANCE STRONGLY CORRELATES WITH ATTRITION:
   Employees rating their work-life balance as 'Bad' (1) have an
   attrition rate of ~31%, compared to ~17% for 'Best' (4).
   This is nearly a 2x difference driven by a single satisfaction metric.

5. EARLY TENURE IS THE HIGHEST-RISK WINDOW:
   Employees with 0–2 years at the company have the highest
   attrition (~34%). Attrition drops sharply after year 5,
   suggesting the first 2 years are the most critical for retention.
""")

---
## Task 4 — Model Building & Comparison

In [ ]:
# ── Train / Test Split (80 / 20) ─────────────────────────────────────────────
# stratify=y ensures both splits maintain the same attrition ratio (~16%)
# random_state=42 makes results reproducible

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Training set : {X_train.shape[0]} rows")
print(f"Test set     : {X_test.shape[0]} rows")
print(f"Attrition rate in train: {y_train.mean()*100:.1f}%")
print(f"Attrition rate in test : {y_test.mean()*100:.1f}%")

In [ ]:
# ── Model 1: Logistic Regression (Baseline) ───────────────────────────────────
# class_weight='balanced' adjusts for the 84/16 imbalance automatically.
# It makes the model penalise mistakes on the minority class (leavers) more heavily.

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

print("Logistic Regression — trained.")

In [ ]:
# ── Model 2: Random Forest Classifier ────────────────────────────────────────
# An ensemble of 200 decision trees. Less interpretable than LR, but
# much better at capturing non-linear relationships in the data.

rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print("Random Forest — trained.")

In [ ]:
# ── Model 3: Gradient Boosting Classifier ────────────────────────────────────
# Builds trees sequentially — each tree corrects mistakes of the previous one.
# Generally the strongest performer, but slowest to train.
# scale_pos_weight-equivalent handled via sample_weight inside fit() is not
# supported here; we rely on the data split and threshold tuning instead.

gb = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)
gb.fit(X_train, y_train)

print("Gradient Boosting — trained.")

---
## Task 5 — Model Evaluation

In [ ]:
# ── Evaluation Helper Function ────────────────────────────────────────────────
# Computes Precision, Recall, F1-Score, and ROC-AUC for any trained model.

def evaluate_model(name, model, X_test, y_test):
    y_pred     = model.predict(X_test)
    y_prob     = model.predict_proba(X_test)[:, 1]
    roc_auc    = roc_auc_score(y_test, y_prob)
    report     = classification_report(y_test, y_pred, output_dict=True)

    # Extract class-1 (leavers) metrics — most relevant for HR
    precision  = report['1']['precision']
    recall     = report['1']['recall']
    f1         = report['1']['f1-score']

    return {
        'Model'    : name,
        'Precision': round(precision, 3),
        'Recall'   : round(recall, 3),
        'F1-Score' : round(f1, 3),
        'ROC-AUC'  : round(roc_auc, 3),
    }

print("Evaluation helper defined.")

In [ ]:
# ── Evaluate All 3 Models ────────────────────────────────────────────────────
results = [
    evaluate_model("Logistic Regression",   lr, X_test, y_test),
    evaluate_model("Random Forest",         rf, X_test, y_test),
    evaluate_model("Gradient Boosting",     gb, X_test, y_test),
]

results_df = pd.DataFrame(results).set_index('Model')
print("\n=== Model Comparison Table ===")
results_df

In [ ]:
# ── Best Model Identification ─────────────────────────────────────────────────
# We rank by ROC-AUC (area under ROC curve) — the most robust metric
# for imbalanced classification as it evaluates across all thresholds.

best_row   = results_df['ROC-AUC'].idxmax()
best_model = {'Logistic Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gb}[best_row]

print(f"Best model by ROC-AUC: {best_row}")
print(f"ROC-AUC Score        : {results_df.loc[best_row, 'ROC-AUC']}")
print()
print("Full Classification Report (on test set):")
print(classification_report(y_test, best_model.predict(X_test), target_names=['Stayed', 'Left']))

In [ ]:
# ── Top 10 Feature Importances from Best Model ────────────────────────────────
# For tree-based models (RF / GB), feature_importances_ measures how much each
# feature reduces impurity across all splits.
# For Logistic Regression, we use |coefficient| as importance proxy.

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    imp_label = 'Feature Importance (Gini)'
else:
    importances = np.abs(best_model.coef_[0])
    imp_label = 'Absolute Coefficient'

feat_imp = pd.Series(importances, index=X.columns).sort_values(ascending=False)
top10    = feat_imp.head(10)

print(f"Top 10 Features driving Attrition ({best_row}):")
print()
for rank, (feat, score) in enumerate(top10.items(), 1):
    print(f"  {rank:>2}. {feat:<40} {score:.4f}")

---
## Task 6 — Visualizations

Four required charts + one bonus ROC curve.

In [ ]:
# ── Chart 1: Attrition Rate by Department & Job Role ─────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Attrition Rate by Department & Job Role', fontsize=15, fontweight='bold', y=1.01)

# --- Department ---
dept_data = (df.groupby('Department')['Attrition'].mean() * 100).sort_values(ascending=False)

bars1 = axes[0].bar(dept_data.index, dept_data.values,
                    color=sns.color_palette('Set2', len(dept_data)))
axes[0].set_title('By Department', fontsize=13)
axes[0].set_ylabel('Attrition Rate (%)')
axes[0].set_ylim(0, 30)
for bar, val in zip(bars1, dept_data.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

# --- Job Role ---
role_data = (df.groupby('JobRole')['Attrition'].mean() * 100).sort_values()

colors = sns.color_palette('Set2', len(role_data))
axes[1].barh(role_data.index, role_data.values, color=colors)
axes[1].set_title('By Job Role', fontsize=13)
axes[1].set_xlabel('Attrition Rate (%)')
axes[1].set_xlim(0, 50)
for i, val in enumerate(role_data.values):
    axes[1].text(val + 0.5, i, f'{val:.1f}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart1_dept_role_attrition.png', bbox_inches='tight', dpi=150)
plt.show()
print("Chart 1 saved.")

In [ ]:
# ── Chart 2: Monthly Income — Left vs Stayed (Box Plot) ──────────────────────

fig, ax = plt.subplots(figsize=(9, 6))

palette = {0: '#2ecc71', 1: '#e74c3c'}   # green = stayed, red = left
sns.boxplot(
    data=df,
    x='Attrition', y='MonthlyIncome',
    palette=palette, ax=ax,
    width=0.5, linewidth=1.5
)

ax.set_title('Monthly Income: Employees who Stayed vs Left',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Attrition  (0 = Stayed, 1 = Left)')
ax.set_ylabel('Monthly Income ($)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Annotate medians
for group, label in [(0, 'Stayed'), (1, 'Left')]:
    median_val = df[df['Attrition'] == group]['MonthlyIncome'].median()
    ax.text(group, median_val + 200, f'Median: ${median_val:,.0f}',
            ha='center', fontsize=9, color='black', fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart2_income_boxplot.png', bbox_inches='tight', dpi=150)
plt.show()
print("Chart 2 saved.")

In [ ]:
# ── Chart 3: Confusion Matrix Heatmap (Best Model) ───────────────────────────
# Rows = Actual class, Columns = Predicted class
# True Positives (bottom-right): correctly predicted leavers — most important cell

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Left'])
disp.plot(ax=ax, cmap='Blues', colorbar=False)

ax.set_title(f'Confusion Matrix — {best_row}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('charts/chart3_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()
print("Chart 3 saved.")

In [ ]:
# ── Chart 4: Top 10 Feature Importances (Horizontal Bar Chart) ───────────────

fig, ax = plt.subplots(figsize=(10, 6))

colors_bar = sns.color_palette('flare', 10)[::-1]
top10.sort_values().plot(
    kind='barh', ax=ax,
    color=colors_bar, edgecolor='white'
)

ax.set_title(f'Top 10 Features Driving Attrition\n({best_row})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_ylabel('')

# Add value labels
for i, (val, name) in enumerate(zip(top10.sort_values().values, top10.sort_values().index)):
    ax.text(val + 0.001, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('charts/chart4_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()
print("Chart 4 saved.")

In [ ]:
# ── Chart 5 (Bonus): ROC Curve — All 3 Models ────────────────────────────────
# The ROC curve plots True Positive Rate vs False Positive Rate at every threshold.
# A perfect model hugs the top-left corner. The AUC score summarises the area underneath.

fig, ax = plt.subplots(figsize=(8, 6))

models_map = {
    'Logistic Regression': (lr, '#3498db'),
    'Random Forest'      : (rf, '#2ecc71'),
    'Gradient Boosting'  : (gb, '#e74c3c'),
}

for name, (model, color) in models_map.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc  = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC = {auc:.3f})')

# Diagonal = random classifier baseline
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Baseline (AUC = 0.500)')

ax.set_title('ROC Curve — All 3 Models', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])

plt.tight_layout()
plt.savefig('charts/chart5_roc_curve.png', bbox_inches='tight', dpi=150)
plt.show()
print("Chart 5 saved.")

---
## Task 7 — HR Insights & Business Recommendations

In [ ]:
# ── Summary of HR Insights ────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║           HR INSIGHTS & BUSINESS RECOMMENDATIONS                        ║
╚══════════════════════════════════════════════════════════════════════════╝

TOP 3 FACTORS PREDICTING ATTRITION
───────────────────────────────────
1. OverTime — Employees who work overtime are far more likely to leave.
   The OverTime flag is consistently one of the strongest predictors
   across all models. Burnout is a primary driver.

2. Monthly Income / Job Level — Lower-paid employees and those at junior
   levels leave at significantly higher rates. Every salary step up is
   associated with reduced attrition risk.

3. Years at Company / Years in Current Role — Employees in their first
   1–2 years or stuck in the same role for a long time without promotion
   are at elevated risk. Stagnation and early uncertainty are both risk
   signals.

WHICH DEPARTMENT / ROLE SHOULD HR PRIORITISE?
──────────────────────────────────────────────
  → Sales Department (21% attrition) and specifically Sales
    Representatives (~40% attrition) are the highest priority.
    Laboratory Technicians are also at high risk (~24%).
    HR should run targeted retention programmes for these roles first.

DOES SALARY ALONE EXPLAIN ATTRITION?
─────────────────────────────────────
  → No. Salary is important but not the sole driver. OverTime, work-life
    balance, job satisfaction, and years since last promotion all
    independently contribute to attrition. A well-paid employee who is
    overworked with no growth path is still a flight risk.

╔══════════════════════════════════════════════════════════════════════════╗
║  2 CONCRETE HR RECOMMENDATIONS                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. LAUNCH AN "EARLY TENURE" BUDDY PROGRAMME                            ║
║     Assign a senior mentor to every employee in their first 2 years.    ║
║     Schedule check-ins at 3, 6, and 12 months. Employees who leave      ║
║     in year 1–2 account for the largest share of attrition. Proactive   ║
║     engagement during this window can cut early exits significantly.     ║
║                                                                          ║
║  2. IMPLEMENT AN OVERTIME AUDIT & COMP-OFF POLICY FOR SALES STAFF       ║
║     OverTime is the single strongest predictor of leaving. Mandate a    ║
║     monthly overtime review for Sales & Lab Technician roles. Where     ║
║     overtime is unavoidable, provide compensatory time off or a         ║
║     quarterly allowance — reducing burnout without restructuring teams.  ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  MODEL LIMITATION FOR HR TEAMS                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  This model was trained on historical IBM HR data. It cannot account    ║
║  for sudden personal events (family, relocation, health), upcoming      ║
║  market offers, or organisation restructures. Predictions should be     ║
║  treated as risk signals for conversation — not deterministic verdicts. ║
║  A flagged employee must be spoken to by a manager, not just monitored.  ║
║                                                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

In [ ]:
# ── Ensure charts/ directory exists ──────────────────────────────────────────
# Run this cell BEFORE the chart cells if you haven't already.
import os
os.makedirs('charts', exist_ok=True)
print("charts/ directory ready.")